<a href="https://colab.research.google.com/github/LegadimaW/Human-Activity-Recognition-ML/blob/main/Member_2/code/Member2_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis
from scipy.signal import find_peaks
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
# Load cleaned data from Member 1
train_clean = pd.read_parquet("/content/drive/MyDrive/Human_Activity_Recognition_ML/Data/train_data_clean.parquet")
test_clean = pd.read_parquet("/content/drive/MyDrive/Human_Activity_Recognition_ML/Data/test_data_clean.parquet")
labels = pd.read_csv("/content/drive/MyDrive/Human_Activity_Recognition_ML/Data/train_labels.csv")

print(train_clean.shape)
print(test_clean.shape)
print(labels.head())

(279300, 16)
(143800, 16)
   Sample_ID  class_label
0          1          1.0
1          2          1.0
2          3          1.0
3          4          1.0
4          5          1.0


In [9]:
#Identify sensor columns
id_col = "Sample_ID"

sensor_cols = [col for col in train_clean.columns if col not in ["Sample_ID", "Time_Step"]]

print(sensor_cols)
print(len(sensor_cols))

['Signal_M', 'Signal_B', 'Signal_J', 'Signal_H', 'Signal_L', 'Signal_F', 'Signal_D', 'Signal_A', 'Signal_G', 'Signal_K', 'Signal_C', 'Signal_E', 'Signal_I', 'Signal_N']
14


In [12]:
#Create feature extraction function
def extract_features(sample):
    features = {}

    for col in sensor_cols:
        signal = sample[col].values

        # Statistical features
        features[f"{col}_mean"] = np.mean(signal)
        features[f"{col}_std"] = np.std(signal)
        features[f"{col}_min"] = np.min(signal)
        features[f"{col}_max"] = np.max(signal)
        features[f"{col}_median"] = np.median(signal)
        features[f"{col}_skew"] = skew(signal)
        features[f"{col}_kurtosis"] = kurtosis(signal)
        features[f"{col}_iqr"] = np.percentile(signal, 75) - np.percentile(signal, 25)

        # Temporal features
        features[f"{col}_slope"] = np.polyfit(np.arange(len(signal)), signal, 1)[0]
        features[f"{col}_zero_crossings"] = np.sum(np.diff(np.sign(signal)) != 0)

        peaks, _ = find_peaks(signal)
        features[f"{col}_num_peaks"] = len(peaks)

        # FFT features
        fft_values = np.abs(np.fft.rfft(signal))
        features[f"{col}_fft_mean"] = np.mean(fft_values)
        features[f"{col}_fft_max"] = np.max(fft_values)
        features[f"{col}_fft_dominant_freq"] = np.argmax(fft_values)

    return features

In [14]:
#Apply feature extraction to train and test
def build_feature_table(df):
    rows = []

    for sample_id, group in df.groupby("Sample_ID"):
        row = {"Sample_ID": sample_id}
        row.update(extract_features(group))
        rows.append(row)

    return pd.DataFrame(rows)

In [15]:
X_train = build_feature_table(train_clean)
X_test = build_feature_table(test_clean)

print(X_train.shape)
print(X_test.shape)

X_train.head()

(2793, 197)
(1438, 197)


,Sample_ID,Signal_M_mean,Signal_M_std,Signal_M_min,Signal_M_max,Signal_M_median,Signal_M_skew,Signal_M_kurtosis,Signal_M_iqr,Signal_M_slope,...,Signal_N_median,Signal_N_skew,Signal_N_kurtosis,Signal_N_iqr,Signal_N_slope,Signal_N_zero_crossings,Signal_N_num_peaks,Signal_N_fft_mean,Signal_N_fft_max,Signal_N_fft_dominant_freq
0,1,0.000035,0.000028,-0.000035,0.000104,0.000042,0.354932,0.135124,0.000036,6.846817e-08,...,0.000036,-0.430310,0.252043,0.000439,2.687949e-07,53,31,0.003042,0.009545,26
1,2,0.000010,0.000024,-0.000031,0.000077,0.000009,0.687250,0.881553,0.000027,-2.004631e-08,...,-0.000040,0.312898,-0.419336,0.000507,3.171134e-06,43,34,0.003288,0.007517,23
2,3,0.000013,0.000030,-0.000041,0.000079,0.000000,0.499349,-0.579467,0.000038,2.863321e-08,...,-0.000070,-0.181965,-0.137991,0.000480,-4.656474e-07,49,35,0.003159,0.008291,28
3,4,0.000018,0.000022,-0.000017,0.000056,0.000006,0.446511,-1.278325,0.000041,7.838834e-08,...,0.000032,-0.227220,-0.205341,0.000442,-1.167773e-06,49,34,0.002927,0.006802,50
4,5,0.000013,0.000028,-0.000040,0.000104,0.000001,0.568250,0.042432,0.000042,-1.233851e-07,...,0.000079,-0.174453,-0.603394,0.000483,-6.020812e-08,41,28,0.003135,0.008516,34


In [16]:
#Add labels to training features

train_features = X_train.merge(labels, on="Sample_ID", how="left")

print(train_features.shape)
train_features.head()

(2793, 198)


,Sample_ID,Signal_M_mean,Signal_M_std,Signal_M_min,Signal_M_max,Signal_M_median,Signal_M_skew,Signal_M_kurtosis,Signal_M_iqr,Signal_M_slope,...,Signal_N_skew,Signal_N_kurtosis,Signal_N_iqr,Signal_N_slope,Signal_N_zero_crossings,Signal_N_num_peaks,Signal_N_fft_mean,Signal_N_fft_max,Signal_N_fft_dominant_freq,class_label
0,1,0.000035,0.000028,-0.000035,0.000104,0.000042,0.354932,0.135124,0.000036,6.846817e-08,...,-0.430310,0.252043,0.000439,2.687949e-07,53,31,0.003042,0.009545,26,1.0
1,2,0.000010,0.000024,-0.000031,0.000077,0.000009,0.687250,0.881553,0.000027,-2.004631e-08,...,0.312898,-0.419336,0.000507,3.171134e-06,43,34,0.003288,0.007517,23,1.0
2,3,0.000013,0.000030,-0.000041,0.000079,0.000000,0.499349,-0.579467,0.000038,2.863321e-08,...,-0.181965,-0.137991,0.000480,-4.656474e-07,49,35,0.003159,0.008291,28,1.0
3,4,0.000018,0.000022,-0.000017,0.000056,0.000006,0.446511,-1.278325,0.000041,7.838834e-08,...,-0.227220,-0.205341,0.000442,-1.167773e-06,49,34,0.002927,0.006802,50,1.0
4,5,0.000013,0.000028,-0.000040,0.000104,0.000001,0.568250,0.042432,0.000042,-1.233851e-07,...,-0.174453,-0.603394,0.000483,-6.020812e-08,41,28,0.003135,0.008516,34,1.0


In [19]:
X_train.to_parquet("/content/drive/MyDrive/Human_Activity_Recognition_ML/Data/X_train_features.parquet", index=False)
X_test.to_parquet("/content/drive/MyDrive/Human_Activity_Recognition_ML/Data/X_test_features.parquet", index=False)
train_features.to_parquet("/content/drive/MyDrive/Human_Activity_Recognition_ML/Data/train_features_with_labels.parquet", index=False)

print("Saved feature files successfully.")

Saved feature files successfully.
